# LeetCode 2083 — Substrings That Begin and End With the Same Letter bằng Python

Link đề: https://leetcode.com/problems/substrings-that-begin-and-end-with-the-same-letter/ · Độ khó: **Medium** · Chủ đề: Hash Table, Math, String, Counting, Prefix Sum

> **Bài Premium.** Đề trong notebook được viết lại theo hiểu biết về bài (nguồn: doocs/leetcode). Nếu có tài khoản Premium, hãy đối chiếu lại đề gốc.

Skeleton LeetCode:

```python
class Solution:
    def numberOfSubstrings(self, s: str) -> int:
        ...
```

Lộ trình:
1. **Sinh mọi xâu con** rồi kiểm tra ký tự đầu/cuối — `O(n³)` (vì tạo xâu).
2. **Duyệt mọi cặp chỉ số `i ≤ j`** và so `s[i] == s[j]` — `O(n²)`.
3. **Đếm theo chữ cái**: chữ `c` xuất hiện `f` lần cho `f(f+1)/2` xâu con — `O(n)`.
4. **Một lượt duyệt với bộ đếm chạy**: tại vị trí `j`, cộng số lần `s[j]` đã xuất hiện tính cả `j` — `O(n)`, `O(26)` bộ nhớ.

## 1. Mục tiêu học

1. Đổi bài "đếm xâu con" thành bài **đếm cặp chỉ số** `(i, j)`, `i ≤ j`, `s[i] = s[j]` — một xâu con được xác định duy nhất bởi hai đầu mút.
2. Dùng công thức tổ hợp `C(f, 2) + f = f(f+1)/2` để đếm số cặp có lặp.
3. Viết lời giải **một lượt** với bộ đếm chạy — mẫu "đếm cặp bằng nhau" dùng lại ở rất nhiều bài.
4. Phân biệt đếm **xâu con theo vị trí** (đề bài) với đếm **xâu con phân biệt**.
5. Mở rộng sang truy vấn đoạn bằng **prefix count 26 chữ cái**.

## 2. Đề bài & ví dụ

Cho xâu `s` chỉ gồm chữ cái thường. Đếm số **xâu con** (liên tiếp, không rỗng) **bắt đầu và kết thúc bằng cùng một ký tự**. Các xâu con ở vị trí khác nhau được tính riêng, kể cả khi nội dung giống nhau.

**Ví dụ 1:** `s = "abcba"` → `7`: 5 xâu độ dài 1, `"bcb"`, `"abcba"`.

**Ví dụ 2:** `s = "abacad"` → `9`: 6 xâu độ dài 1, `"aba"`, `"aca"`, `"abaca"`.

**Ví dụ 3:** `s = "a"` → `1`.

**Ràng buộc:** `1 ≤ len(s) ≤ 10⁵`, `s` chỉ gồm chữ cái tiếng Anh viết thường.

In [1]:
import random
import string
import time
from collections import Counter
from typing import List

examples = [("abcba", 7), ("abacad", 9), ("a", 1)]
for s, want in examples:
    print(f"s = {s!r} -> kỳ vọng {want}")

s = 'abcba' -> kỳ vọng 7
s = 'abacad' -> kỳ vọng 9
s = 'a' -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1 Xâu con ↔ cặp chỉ số

Mỗi xâu con không rỗng của `s` ứng với đúng một cặp `(i, j)`, `0 ≤ i ≤ j < n`: `s[i:j+1]`. Có `n(n+1)/2` xâu con tất cả.

In [2]:
s = "abc"
subs = [(i, j, s[i:j + 1]) for i in range(len(s)) for j in range(i, len(s))]
print(subs)
print("số xâu con =", len(subs), "= n(n+1)/2 =", len(s) * (len(s) + 1) // 2)

[(0, 0, 'a'), (0, 1, 'ab'), (0, 2, 'abc'), (1, 1, 'b'), (1, 2, 'bc'), (2, 2, 'c')]
số xâu con = 6 = n(n+1)/2 = 6


### 3.2 `Counter` — đếm tần suất

In [3]:
print(Counter("abacad"))
print(Counter("abacad").most_common(2))

Counter({'a': 3, 'b': 1, 'c': 1, 'd': 1})
[('a', 3), ('b', 1)]


### 3.3 Đếm cặp có lặp: `f(f+1)/2`

Từ `f` vị trí, số cặp `(i, j)` với `i ≤ j` = số cặp `i < j` (`C(f, 2) = f(f-1)/2`) + số cặp `i = j` (`f`) = `f(f+1)/2`.

In [4]:
from itertools import combinations_with_replacement
for f in range(1, 6):
    pairs = list(combinations_with_replacement(range(f), 2))
    print(f"f = {f}: {len(pairs)} cặp i <= j, công thức f(f+1)/2 = {f * (f + 1) // 2}")

f = 1: 1 cặp i <= j, công thức f(f+1)/2 = 1
f = 2: 3 cặp i <= j, công thức f(f+1)/2 = 3
f = 3: 6 cặp i <= j, công thức f(f+1)/2 = 6
f = 4: 10 cặp i <= j, công thức f(f+1)/2 = 10
f = 5: 15 cặp i <= j, công thức f(f+1)/2 = 15


### 3.4 Mảng 26 phần tử thay cho dict

`ord(c) - ord('a')` đưa chữ cái về chỉ số `0..25`. Mảng list nhanh hơn `Counter` một chút với vòng lặp Python.

In [5]:
cnt = [0] * 26
for c in "abacad":
    cnt[ord(c) - 97] += 1
print({chr(97 + k): v for k, v in enumerate(cnt) if v})

{'a': 3, 'b': 1, 'c': 1, 'd': 1}


### 3.5 Ma trận cặp `(i, j)` — nhìn đáp án như một tam giác

Vẽ bảng `n × n`, ô `(i, j)` với `i ≤ j` là một xâu con. Đánh dấu `#` nếu `s[i] == s[j]`. Đáp án = số dấu `#` ở **nửa trên** (tính cả đường chéo). Đường chéo luôn toàn `#` (xâu độ dài 1).

In [6]:
def pair_matrix(s):
    n = len(s)
    print("     " + "  ".join(s))
    for i in range(n):
        row = []
        for j in range(n):
            row.append(" " if j < i else ("#" if s[i] == s[j] else "."))
        print(f"{i} {s[i]} |" + "  ".join(row))
    print("số # =", sum(1 for i in range(n) for j in range(i, n) if s[i] == s[j]))

pair_matrix("abcba")

     a  b  c  b  a
0 a |#  .  .  .  #
1 b |   #  .  #  .
2 c |      #  .  .
3 b |         #  .
4 a |            #
số # = 7


Đọc theo **cột** `j`: số `#` trong cột `j` = số xâu con hợp lệ **kết thúc tại `j`** — đó chính là thứ Cách 4 cộng ở mỗi bước. Đọc theo **hàng** `i`: số xâu con hợp lệ bắt đầu tại `i` — cách đếm "từ phải sang trái" cũng cho cùng tổng.

## 4. Ý tưởng nền tảng / trực giác

Liệt kê cặp đầu mút cho `s = "abacad"` (chỉ số `0..5`):

| chữ | vị trí | cặp `(i, j)`, `i ≤ j` | số cặp |
|---|---|---|---|
| a | 0, 2, 4 | (0,0) (2,2) (4,4) (0,2) (0,4) (2,4) | 6 = 3·4/2 |
| b | 1 | (1,1) | 1 |
| c | 3 | (3,3) | 1 |
| d | 5 | (5,5) | 1 |

Tổng `6 + 1 + 1 + 1 = 9`. ✓ Các cặp `(0,2)`, `(2,4)`, `(0,4)` chính là `"aba"`, `"aca"`, `"abaca"`.

**Điều quan trọng:** ký tự **ở giữa** không ảnh hưởng gì. Chỉ cần biết **mỗi chữ xuất hiện bao nhiêu lần**.

In [7]:
s = "abacad"
for ch in sorted(set(s)):
    ps = [i for i, c in enumerate(s) if c == ch]
    pairs = [(i, j) for a, i in enumerate(ps) for j in ps[a:]]
    print(ch, ps, "->", [s[i:j + 1] for i, j in pairs])

a [0, 2, 4] -> ['a', 'aba', 'abaca', 'a', 'aca', 'a']
b [1] -> ['b']
c [3] -> ['c']
d [5] -> ['d']


## 5. Cách 1 — Brute Force / baseline: sinh mọi xâu con

**Ý tưởng:** đúng nghĩa đen — cắt `s[i:j+1]` và kiểm tra `sub[0] == sub[-1]`.

In [8]:
def count_brute_substrings(s):
    cnt = 0
    for i in range(len(s)):
        for j in range(i, len(s)):
            sub = s[i:j + 1]                 # tạo xâu tốn O(j - i)
            if sub[0] == sub[-1]:
                cnt += 1
    return cnt

for s, want in examples:
    print(count_brute_substrings(s), "kỳ vọng", want)

7 kỳ vọng 7
9 kỳ vọng 9
1 kỳ vọng 1


**Độ phức tạp:** `O(n²)` xâu con × `O(n)` để tạo mỗi xâu ⇒ `O(n³)` thời gian; `O(n)` bộ nhớ tạm. Với `n = 10⁵` hoàn toàn không khả thi.

**Nút thắt 1:** không cần **tạo** xâu con — chỉ cần hai đầu mút `s[i]`, `s[j]`.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** đáp án = số cặp `(i, j)` với `i ≤ j` và `s[i] = s[j]`. Nhóm theo chữ cái:

`đáp án = Σ_{c} f_c (f_c + 1) / 2`, với `f_c` = số lần chữ `c` xuất hiện.

**Phiên bản một lượt:** duyệt `j` từ trái sang phải, giữ `cnt[c]` = số lần `c` xuất hiện trong `s[0..j]`. Số xâu con hợp lệ **kết thúc tại `j`** = số `i ≤ j` có `s[i] = s[j]` = `cnt[s[j]]` (sau khi đã tăng cho chính `j`).

**Invariant:** sau khi xử lý `j`, `ans` = số xâu con hợp lệ nằm trong `s[0..j]`.

Quy tắc: *nếu* bài hỏi "đếm cặp/xâu con mà hai đầu thỏa điều kiện chỉ phụ thuộc giá trị tại đầu mút" *thì* đếm tần suất và dùng tổ hợp (hoặc bộ đếm chạy).

**Đếm theo đầu trái (đối xứng):** duyệt `i` từ **phải sang trái**, `cnt[c]` = số lần `c` trong `s[i..n-1]`; số xâu hợp lệ **bắt đầu tại `i`** là `cnt[s[i]]` sau khi tăng. Hai cách duyệt đếm cùng một tập cặp, chỉ khác "gán" mỗi cặp cho đầu nào — một kỹ thuật kiểm chứng hữu ích: nếu hai chiều cho kết quả khác nhau thì code có lỗi.

In [9]:
def count_from_right(s):
    cnt, ans, starts = Counter(), 0, []
    for i in range(len(s) - 1, -1, -1):
        cnt[s[i]] += 1; ans += cnt[s[i]]; starts.append(cnt[s[i]])
    return ans, starts[::-1]

for s_, want in examples:
    a, st = count_from_right(s_)
    print(f"{s_!r}: số xâu hợp lệ bắt đầu tại mỗi i = {st} -> tổng {a} (kỳ vọng {want})")

'abcba': số xâu hợp lệ bắt đầu tại mỗi i = [2, 2, 1, 1, 1] -> tổng 7 (kỳ vọng 7)
'abacad': số xâu hợp lệ bắt đầu tại mỗi i = [3, 1, 2, 1, 1, 1] -> tổng 9 (kỳ vọng 9)
'a': số xâu hợp lệ bắt đầu tại mỗi i = [1] -> tổng 1 (kỳ vọng 1)


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Duyệt mọi cặp chỉ số (không tạo xâu)

In [10]:
def count_pairs_n2(s):
    n, cnt = len(s), 0
    for i in range(n):
        for j in range(i, n):
            if s[i] == s[j]:
                cnt += 1
    return cnt

for s, want in examples:
    print(count_pairs_n2(s), "kỳ vọng", want)

7 kỳ vọng 7
9 kỳ vọng 9
1 kỳ vọng 1


**Độ phức tạp:** `O(n²)` thời gian (`5·10⁹` cặp với `n = 10⁵` — vẫn quá chậm), `O(1)` bộ nhớ. Đây là "trọng tài" tốt cho test ngẫu nhiên.

Tối ưu nhỏ: với mỗi `i`, đếm `s[i]` trong `s[i:]` bằng `s.count(s[i], i)` (chạy trong C) — vẫn `O(n²)` nhưng nhanh hơn hàng chục lần.

### 7.2 Cách 3 — Đếm theo chữ cái + công thức

In [11]:
def count_formula(s):
    return sum(f * (f + 1) // 2 for f in Counter(s).values())

for s, want in examples:
    print(count_formula(s), "kỳ vọng", want, "| tần suất:", dict(Counter(s)))

7 kỳ vọng 7 | tần suất: {'a': 2, 'b': 2, 'c': 1}
9 kỳ vọng 9 | tần suất: {'a': 3, 'b': 1, 'c': 1, 'd': 1}
1 kỳ vọng 1 | tần suất: {'a': 1}


**Độ phức tạp:** `O(n)` để đếm + `O(26)` để cộng ⇒ `O(n)`. Space `O(26)`.

### 7.3 Cách 4 — Một lượt với bộ đếm chạy (code nộp)

In [12]:
def count_running(s):
    cnt = [0] * 26
    ans = 0
    for c in s:
        k = ord(c) - 97
        cnt[k] += 1              # tính cả xâu con độ dài 1 kết thúc tại đây
        ans += cnt[k]            # mọi i <= j có s[i] == s[j]
    return ans

for s, want in examples:
    print(count_running(s), "kỳ vọng", want)

7 kỳ vọng 7
9 kỳ vọng 9
1 kỳ vọng 1


Code nộp LeetCode của Cách 4: xem mục 9.

**Độ phức tạp:** `O(n)` thời gian, `O(26)` bộ nhớ. Ưu điểm so với Cách 3: xử lý được **luồng dữ liệu** (stream) — biết đáp án của mọi tiền tố.

**Vì sao hai cách cho cùng kết quả?** Với chữ `c` xuất hiện `f` lần, Cách 4 cộng `1 + 2 + … + f = f(f+1)/2` — đúng biểu thức của Cách 3.

In [13]:
s = "abacabaa"
prefix_answers, cnt, ans = [], Counter(), 0
for c in s:
    cnt[c] += 1; ans += cnt[c]; prefix_answers.append(ans)
print("đáp án mọi tiền tố:", prefix_answers)
print("kiểm tra:", [count_formula(s[:k]) for k in range(1, len(s) + 1)])

đáp án mọi tiền tố: [1, 2, 4, 5, 8, 10, 14, 19]
kiểm tra: [1, 2, 4, 5, 8, 10, 14, 19]


### 7.4 Cách 5 — DP theo lần xuất hiện trước đó

Gọi `end[j]` = số xâu con hợp lệ **kết thúc tại `j`**. Nếu `p = prev[j]` là vị trí gần nhất trước `j` có cùng chữ thì mọi xâu con hợp lệ kết thúc tại `p` "kéo dài" tới `j` vẫn hợp lệ (đầu trái giữ nguyên, đầu phải đổi sang `j` — cùng chữ), cộng thêm xâu `s[j..j]`:

`end[j] = end[p] + 1` (nếu có `p`), ngược lại `end[j] = 1`. Đáp án `= Σ end[j]`.

Cách này không dùng tổ hợp — chỉ dùng quan hệ truy hồi giữa hai lần xuất hiện liên tiếp. Nó là khuôn cho các bài "đếm mảng con kết thúc tại `j`" mà điều kiện phụ thuộc lần xuất hiện trước.

In [14]:
def count_dp_prev(s):
    last = {}                    # chữ -> vị trí gần nhất
    end = [0] * len(s)
    for j, c in enumerate(s):
        end[j] = (end[last[c]] + 1) if c in last else 1
        last[c] = j
    return sum(end)

for s_, want in examples:
    print(count_dp_prev(s_), "kỳ vọng", want)

7 kỳ vọng 7
9 kỳ vọng 9
1 kỳ vọng 1


**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ cho `end` (có thể giảm về `O(26)` bằng cách lưu `end` theo chữ thay vì theo vị trí — khi đó nó trùng hẳn với `cnt[c]` của Cách 4).

**Liên hệ:** với chữ `c`, `end` tại lần xuất hiện thứ `k` của `c` bằng `k` ⇒ `end[j] = cnt[s[j]]` sau khi tăng. Ba cách 3, 4, 5 là ba góc nhìn của cùng một phép đếm.

## 8. TRACE từng bước

Trace Cách 4 trên `"abcba"` và `"abacad"`: in `j`, ký tự, `cnt[c]` sau khi tăng, phần cộng thêm, `ans`, và các xâu con hợp lệ kết thúc tại `j`.

In [15]:
def trace_running(s):
    cnt, ans = Counter(), 0
    print(f"s = {s!r}")
    print(f"{'j':>2} | c | cnt[c] | +  | ans | xâu con kết thúc tại j")
    for j, c in enumerate(s):
        cnt[c] += 1; ans += cnt[c]
        ends = [s[i:j + 1] for i in range(j + 1) if s[i] == c]
        print(f"{j:>2} | {c} | {cnt[c]:>6} | {cnt[c]:>2} | {ans:>3} | {ends}")
    print()
    return ans

trace_running("abcba")
trace_running("abacad")

s = 'abcba'
 j | c | cnt[c] | +  | ans | xâu con kết thúc tại j
 0 | a |      1 |  1 |   1 | ['a']
 1 | b |      1 |  1 |   2 | ['b']
 2 | c |      1 |  1 |   3 | ['c']
 3 | b |      2 |  2 |   5 | ['bcb', 'b']
 4 | a |      2 |  2 |   7 | ['abcba', 'a']

s = 'abacad'
 j | c | cnt[c] | +  | ans | xâu con kết thúc tại j
 0 | a |      1 |  1 |   1 | ['a']
 1 | b |      1 |  1 |   2 | ['b']
 2 | a |      2 |  2 |   4 | ['aba', 'a']
 3 | c |      1 |  1 |   5 | ['c']
 4 | a |      3 |  3 |   8 | ['abaca', 'aca', 'a']
 5 | d |      1 |  1 |   9 | ['d']



9

Bảng `"abcba"` tóm tắt: các phần cộng `1, 1, 1, 2, 2` ⇒ `7`. Ở `j = 3` (`'b'`): xâu con `"b"` và `"bcb"`. Ở `j = 4` (`'a'`): `"a"` và `"abcba"`. ✓

In [16]:
def trace_formula(s):
    f = Counter(s); total = 0
    for c in sorted(f):
        add = f[c] * (f[c] + 1) // 2; total += add
        print(f"chữ {c!r}: f = {f[c]} -> f(f+1)/2 = {add}   (tổng {total})")
    return total

trace_formula("mississippi")
print("đối chiếu brute:", count_pairs_n2("mississippi"))

chữ 'i': f = 4 -> f(f+1)/2 = 10   (tổng 10)
chữ 'm': f = 1 -> f(f+1)/2 = 1   (tổng 11)
chữ 'p': f = 2 -> f(f+1)/2 = 3   (tổng 14)
chữ 's': f = 4 -> f(f+1)/2 = 10   (tổng 24)
đối chiếu brute: 24


In [17]:
def trace_dp_prev(s):
    last, end = {}, [0] * len(s)
    print(f"{'j':>2} | c | prev | end[prev] | end[j] | tổng")
    tot = 0
    for j, c in enumerate(s):
        p = last.get(c, -1)
        end[j] = end[p] + 1 if p >= 0 else 1
        last[c] = j; tot += end[j]
        print(f"{j:>2} | {c} | {p:>4} | {end[p] if p >= 0 else '-':>9} | {end[j]:>6} | {tot}")
    return tot

trace_dp_prev("abacabad")
print("đối chiếu:", count_pairs_n2("abacabad"))

 j | c | prev | end[prev] | end[j] | tổng
 0 | a |   -1 |         - |      1 | 1
 1 | b |   -1 |         - |      1 | 2
 2 | a |    0 |         1 |      2 | 4
 3 | c |   -1 |         - |      1 | 5
 4 | a |    2 |         2 |      3 | 8
 5 | b |    1 |         1 |      2 | 10
 6 | a |    4 |         3 |      4 | 14
 7 | d |   -1 |         - |      1 | 15
đối chiếu: 15


## 9. Code nộp LeetCode

In [18]:
class Solution:
    def numberOfSubstrings(self, s: str) -> int:
        cnt = [0] * 26
        ans = 0
        for c in s:
            k = ord(c) - 97
            cnt[k] += 1
            ans += cnt[k]
        return ans

sol = Solution()
for s, want in examples:
    got = sol.numberOfSubstrings(s)
    print(got, "OK" if got == want else "SAI")

7 OK
9 OK
1 OK


## 10. Vì sao thuật toán đúng

**Bổ đề 1 (song ánh).** Ánh xạ `(i, j) ↦ s[i..j]` là song ánh giữa tập cặp `0 ≤ i ≤ j < n` và tập xâu con **theo vị trí**. Xâu con bắt đầu và kết thúc cùng ký tự ⇔ `s[i] = s[j]`. Vậy đáp án `= |{(i, j) : i ≤ j, s[i] = s[j]}|`.

**Bổ đề 2 (phân hoạch theo chữ).** Các cặp trên được phân hoạch theo chữ `c = s[i] = s[j]`. Với chữ `c` có tập vị trí `P_c` (`|P_c| = f_c`), số cặp `i ≤ j` trong `P_c` là `f_c(f_c + 1)/2` (mục 3.3). ⇒ Cách 3 đúng.

**Cách 4 (invariant).** Sau khi xử lý `j`, `cnt[c] = |P_c ∩ [0, j]|`. Số cặp có `j` là đầu phải = `|{i ≤ j : s[i] = s[j]}| = cnt[s[j]]` (đã tính cả `i = j`). Cộng trên mọi `j` đếm mỗi cặp đúng một lần (theo đầu phải). ✓

**Cách 5 (truy hồi).** Gọi `P = prev[j]`. Tập xâu con hợp lệ kết thúc tại `j` gồm `s[j..j]` và các `s[i..j]` với `i < j`, `s[i] = s[j]`. Với `i < j` như vậy thì `i ≤ P` (vì `P` là vị trí cùng chữ gần nhất trước `j`), và `s[i] = s[P]` ⇒ `s[i..P]` là xâu hợp lệ kết thúc tại `P`. Ngược lại mỗi xâu hợp lệ `s[i..P]` cho `s[i..j]` hợp lệ. Song ánh ⇒ `end[j] = end[P] + 1`. ✓

**Chặn trên:** đáp án tối đa khi mọi ký tự giống nhau: `n(n+1)/2 = 5 000 050 000` với `n = 10⁵` — vượt `int32`; Python không lo, nhưng ở C++/Java phải dùng `long`.

## 11. Phân tích Time / Space

`n = len(s) ≤ 10⁵`, `Σ = 26`.

| Cách | Time | Space |
|---|---|---|
| 1. Sinh xâu con | `O(n³)` | `O(n)` |
| 2. Mọi cặp | `O(n²)` | `O(1)` |
| 3. Counter + công thức | `O(n + Σ)` | `O(Σ)` |
| 4. Bộ đếm chạy | `O(n)` | `O(Σ)` |

In [19]:
rng = random.Random(1)
big = "".join(rng.choice(string.ascii_lowercase) for _ in range(10**5))
same = "z" * 10**5
for name, fn in [("formula", count_formula), ("running", count_running), ("submit", Solution().numberOfSubstrings)]:
    t0 = time.perf_counter(); a = fn(big); b = fn(same)
    print(f"{name:8s}: random = {a}, toàn 'z' = {b}  ({(time.perf_counter() - t0) * 1000:.1f} ms)")
print("n(n+1)/2 =", 10**5 * (10**5 + 1) // 2, "> 2^31 - 1 =", 2**31 - 1)

t0 = time.perf_counter(); count_pairs_n2(big[:2000]); print(f"O(n^2) với n = 2000: {time.perf_counter() - t0:.2f}s")

formula : random = 192399461, toàn 'z' = 5000050000  (4.5 ms)
running : random = 192399461, toàn 'z' = 5000050000  (10.1 ms)
submit  : random = 192399461, toàn 'z' = 5000050000  (10.0 ms)
n(n+1)/2 = 5000050000 > 2^31 - 1 = 2147483647
O(n^2) với n = 2000: 0.05s


**Đọc kết quả đo:** Cách 3 (`Counter(s)`) nhanh hơn Cách 4 khoảng 2 lần dù cùng `O(n)`, vì `Counter` đếm bằng vòng lặp viết trong C, còn Cách 4 chạy vòng `for` Python với vài phép toán mỗi ký tự. Ở `n = 10⁵` cả hai đều chỉ vài mili-giây — chênh lệch không quan trọng; chọn cách nào dễ đọc và dễ chứng minh hơn. Còn Cách 2 (`O(n²)`) ở `n = 2000` đã mất cỡ `0.05s`, nhân `2500` lần cho `n = 10⁵` ⇒ hàng phút — đó là lý do phải có lời giải tuyến tính.

## 12. Bộ test

In [20]:
fns = [count_pairs_n2, count_formula, count_running, Solution().numberOfSubstrings]

for s, want in examples:
    assert count_brute_substrings(s) == want
    for fn in fns:
        assert fn(s) == want

edge = [("a", 1), ("aa", 3), ("ab", 2), ("aaa", 6), ("abab", 6), ("abcdefghijklmnopqrstuvwxyz", 26),
        ("zzzzz", 15), ("abba", 6)]
for s, want in edge:
    assert count_brute_substrings(s) == want, s
    for fn in fns:
        assert fn(s) == want, (fn, s)

rng = random.Random(2024)
for _ in range(1000):
    k = rng.choice([1, 2, 3, 26])
    s = "".join(rng.choice(string.ascii_lowercase[:k]) for _ in range(rng.randint(1, 40)))
    want = count_brute_substrings(s)
    for fn in fns:
        assert fn(s) == want, (fn, s)
print("ví dụ + edge + 1000 xâu ngẫu nhiên khớp brute force")

ví dụ + edge + 1000 xâu ngẫu nhiên khớp brute force


**Test có cấu trúc (đáp án biết trước bằng công thức):**

| xâu | tần suất | đáp án |
|---|---|---|
| `"ab" * k` | a = k, b = k | `2 · k(k+1)/2 = k(k+1)` |
| 26 chữ khác nhau lặp `m` lần | mỗi chữ `m` | `26 · m(m+1)/2` |
| `"a" * n` | a = n | `n(n+1)/2` |

Các test này kiểm tra cả **độ lớn** của kết quả (vượt `2³¹`) và tính đúng trên xâu dài mà brute force không chạy nổi.

In [21]:
for k in (1, 10, 1000, 50000):
    assert Solution().numberOfSubstrings("ab" * k) == k * (k + 1)
for m in (1, 7, 3846):
    assert Solution().numberOfSubstrings(string.ascii_lowercase * m) == 26 * m * (m + 1) // 2
assert Solution().numberOfSubstrings("a" * 10**5) == 10**5 * (10**5 + 1) // 2
# đảo ngược xâu không đổi đáp án (đếm theo đầu trái hay đầu phải đều như nhau)
rng = random.Random(77)
for _ in range(200):
    s_ = "".join(rng.choice("abcde") for _ in range(rng.randint(1, 200)))
    assert count_running(s_) == count_running(s_[::-1]) == count_dp_prev(s_)
print("test cấu trúc + bất biến khi đảo xâu: OK")

test cấu trúc + bất biến khi đảo xâu: OK


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Quên xâu con độ dài 1: dùng `f(f-1)/2`

In [22]:
def wrong_c2(s):
    return sum(f * (f - 1) // 2 for f in Counter(s).values())    # SAI: chỉ đếm cặp i < j

for s, want in examples:
    print(f"{s!r}: sai = {wrong_c2(s)} | đúng = {want}")

'abcba': sai = 2 | đúng = 7
'abacad': sai = 3 | đúng = 9
'a': sai = 0 | đúng = 1


Xâu con độ dài 1 cũng "bắt đầu và kết thúc cùng ký tự" (ví dụ 1 đếm cả `"a"`, `"b"`, …). **Sửa:** `f(f+1)/2`, hoặc cộng thêm `n`.

### Lỗi 2 — Cộng trước khi tăng bộ đếm

In [23]:
def wrong_add_before_inc(s):
    cnt, ans = Counter(), 0
    for c in s:
        ans += cnt[c]             # SAI: chưa tính chính vị trí hiện tại
        cnt[c] += 1
    return ans

for s, want in examples:
    print(f"{s!r}: sai = {wrong_add_before_inc(s)} | đúng = {want}")

'abcba': sai = 2 | đúng = 7
'abacad': sai = 3 | đúng = 9
'a': sai = 0 | đúng = 1


Thứ tự "cộng rồi mới tăng" đếm cặp `i < j` (mẫu của bài *Number of Good Pairs* 1512) — thiếu đúng `n` xâu độ dài 1. **Sửa:** tăng trước, cộng sau (hoặc `ans + len(s)`).

### Lỗi 3 — Đếm xâu con **phân biệt** (dùng `set`)

In [24]:
def wrong_distinct(s):
    seen = set()
    for i in range(len(s)):
        for j in range(i, len(s)):
            if s[i] == s[j]:
                seen.add(s[i:j + 1])      # SAI: gộp các xâu con giống nội dung ở vị trí khác
    return len(seen)

for s, want in examples + [("aaa", 6), ("abab", 6)]:
    print(f"{s!r}: sai = {wrong_distinct(s)} | đúng = {want}")

'abcba': sai = 5 | đúng = 7
'abacad': sai = 7 | đúng = 9
'a': sai = 1 | đúng = 1
'aaa': sai = 3 | đúng = 6
'abab': sai = 4 | đúng = 6


Đề đếm theo **vị trí**: `"abcba"` có hai xâu `"a"` (vị trí 0 và 4) được tính riêng. **Sửa:** đếm cặp chỉ số, không đếm nội dung.

### Lỗi 4 — Chỉ đếm cặp chữ **liền kề** cùng nhau

In [25]:
def wrong_adjacent_only(s):
    cnt = len(s)
    for i in range(len(s) - 1):
        if s[i] == s[i + 1]:
            cnt += 1              # SAI: chỉ thấy "aa", bỏ "aba", "abcba", ...
    return cnt

for s, want in examples + [("aaa", 6)]:
    print(f"{s!r}: sai = {wrong_adjacent_only(s)} | đúng = {want}")

'abcba': sai = 5 | đúng = 7
'abacad': sai = 6 | đúng = 9
'a': sai = 1 | đúng = 1
'aaa': sai = 5 | đúng = 6


Hai đầu mút có thể cách xa nhau tùy ý; ký tự ở giữa không quan trọng. **Sửa:** đếm mọi cặp vị trí cùng chữ.

### Lỗi 5 — `s.count(c)` trong vòng lặp (đúng nhưng `O(26 n)` hoặc tệ hơn)

Viết `sum(s.count(c) * (s.count(c) + 1) // 2 for c in s)` — vừa **sai** (mỗi chữ cộng lặp `f` lần vì duyệt `for c in s`) vừa `O(n²)`. **Sửa:** duyệt `set(s)` hoặc dùng `Counter`.

In [26]:
def wrong_count_in_loop(s):
    return sum(s.count(c) * (s.count(c) + 1) // 2 for c in s)          # SAI: lặp theo từng ký tự

def ok_count_set(s):
    return sum(s.count(c) * (s.count(c) + 1) // 2 for c in set(s))     # đúng (O(26 n))

for s, want in examples:
    print(f"{s!r}: sai = {wrong_count_in_loop(s)} | duyệt set = {ok_count_set(s)} | đúng = {want}")

'abcba': sai = 13 | duyệt set = 7 | đúng = 7
'abacad': sai = 21 | duyệt set = 9 | đúng = 9
'a': sai = 1 | duyệt set = 1 | đúng = 1


### Lỗi 6 — Truy hồi quên `+1` hoặc dùng `prev` sai (lần xuất hiện **đầu tiên** thay vì gần nhất)

In [27]:
def wrong_dp_first(s):
    first, end = {}, [0] * len(s)
    for j, c in enumerate(s):
        if c in first:
            end[j] = end[first[c]] + 1          # SAI: phải là lần xuất hiện GẦN NHẤT
        else:
            end[j] = 1; first[c] = j
    return sum(end)

def wrong_dp_no_plus1(s):
    last, end = {}, [0] * len(s)
    for j, c in enumerate(s):
        end[j] = end[last[c]] if c in last else 1   # SAI: thiếu xâu s[j..j]
        last[c] = j
    return sum(end)

for s_ in ["abcba", "abacad", "aaaa", "abab"]:
    print(f"{s_!r}: dùng first = {wrong_dp_first(s_)} | thiếu +1 = {wrong_dp_no_plus1(s_)} | đúng = {count_running(s_)}")

'abcba': dùng first = 7 | thiếu +1 = 5 | đúng = 7
'abacad': dùng first = 8 | thiếu +1 = 6 | đúng = 9
'aaaa': dùng first = 7 | thiếu +1 = 4 | đúng = 10
'abab': dùng first = 6 | thiếu +1 = 4 | đúng = 6


Với `"aaaa"`: `end` đúng là `1, 2, 3, 4` (tổng `10`); dùng lần đầu tiên cho `1, 2, 2, 2` (tổng `7`); thiếu `+1` cho `1, 1, 1, 1` (tổng `4`). **Sửa:** cập nhật `last[c] = j` ở **mỗi** lần gặp và cộng `1`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đếm xâu con / mảng con mà điều kiện chỉ phụ thuộc **hai đầu mút**" ⇒ đếm **cặp chỉ số**, nhóm theo giá trị.
- Cặp `i < j` ⇒ "cộng rồi tăng"; cặp `i ≤ j` ⇒ "tăng rồi cộng". Đây là cặp mẫu nên thuộc.
- Kết quả có thể lớn (`n²/2`) — ở ngôn ngữ khác cần kiểu 64 bit.
- Mở rộng "trên đoạn `[l, r]`" ⇒ prefix count theo 26 chữ cái.
- Họ bài: 1512 *Number of Good Pairs*, 1638 *Count Substrings That Differ by One Character*, 2001 *Number of Pairs of Interchangeable Rectangles*, 2588 *Count the Number of Beautiful Subarrays* (cùng mẫu đếm bằng bộ đếm chạy trên prefix XOR).

## 15. Biến thể của bài

1. **Độ dài ≥ 2** (bỏ xâu độ dài 1): `Σ C(f, 2)`.
2. **Truy vấn đoạn** `[l, r]`: tần suất mỗi chữ trong đoạn qua prefix 26 chữ ⇒ `O(26)` mỗi truy vấn (Bài 5).
3. **Tổng độ dài** các xâu con hợp lệ: với vị trí `p_1 < … < p_f` của một chữ, cộng `Σ_{a ≤ b} (p_b - p_a + 1)` — tính bằng prefix sum vị trí (Bài 4).
4. **Đầu và cuối thuộc cùng "nhóm"** (ví dụ cùng là nguyên âm): thay chữ cái bằng nhãn nhóm rồi áp dụng nguyên công thức.
5. **Đầu khác cuối:** `n(n+1)/2 - đáp án`.

In [28]:
def at_least_two(s):
    return sum(f * (f - 1) // 2 for f in Counter(s).values())

def vowel_group(s):
    """Biến thể 4: đầu và cuối cùng là nguyên âm hoặc cùng là phụ âm."""
    lab = ["V" if c in "aeiou" else "C" for c in s]
    return sum(f * (f + 1) // 2 for f in Counter(lab).values())

def vowel_group_brute(s):
    v = lambda c: c in "aeiou"
    return sum(1 for i in range(len(s)) for j in range(i, len(s)) if v(s[i]) == v(s[j]))

def different_ends(s):
    n = len(s); return n * (n + 1) // 2 - count_running(s)

rng = random.Random(15)
for _ in range(300):
    s = "".join(rng.choice("abeiz") for _ in range(rng.randint(1, 15)))
    assert vowel_group(s) == vowel_group_brute(s)
    assert at_least_two(s) == count_pairs_n2(s) - len(s)
    assert different_ends(s) == sum(1 for i in range(len(s)) for j in range(i, len(s)) if s[i] != s[j])
print("abacad: độ dài >= 2 =", at_least_two("abacad"), "| nhóm nguyên âm/phụ âm =", vowel_group("abacad"),
      "| đầu khác cuối =", different_ends("abacad"))

abacad: độ dài >= 2 = 3 | nhóm nguyên âm/phụ âm = 12 | đầu khác cuối = 12


**Biến thể 6 — giới hạn độ dài `≤ L`:** đếm cặp `i ≤ j`, `s[i] = s[j]`, `j - i + 1 ≤ L`. Với mỗi `j`, số `i` hợp lệ = số lần chữ `s[j]` xuất hiện trong cửa sổ `[j - L + 1, j]`. Duy trì mảng đếm 26 chữ trên **cửa sổ trượt** độ dài `L` ⇒ `O(n)`.

In [29]:
def bounded_length(s, L):
    cnt, ans = [0] * 26, 0
    for j, c in enumerate(s):
        cnt[ord(c) - 97] += 1
        if j - L >= 0:
            cnt[ord(s[j - L]) - 97] -= 1          # ký tự rời khỏi cửa sổ
        ans += cnt[ord(c) - 97]
    return ans

def bounded_length_brute(s, L):
    return sum(1 for i in range(len(s)) for j in range(i, min(len(s), i + L)) if s[i] == s[j])

rng = random.Random(16)
for _ in range(500):
    s_ = "".join(rng.choice("abc") for _ in range(rng.randint(1, 20))); L = rng.randint(1, 22)
    assert bounded_length(s_, L) == bounded_length_brute(s_, L)
print("độ dài <= L, 'abacad':", [bounded_length("abacad", L) for L in range(1, 7)], "(L = 6 trùng đáp án gốc 9)")

độ dài <= L, 'abacad': [6, 6, 8, 8, 9, 9] (L = 6 trùng đáp án gốc 9)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Sinh xâu con | nghĩa đen | `O(n³)` | `O(n)` | minh họa |
| 2. Mọi cặp | không tạo xâu | `O(n²)` | `O(1)` | trọng tài cho test |
| 3. Counter + `f(f+1)/2` | tổ hợp | `O(n)` | `O(Σ)` | gọn nhất |
| 4. Bộ đếm chạy | đếm theo đầu phải | `O(n)` | `O(Σ)` | **nộp bài**, xử lý luồng |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `"banana"`? `"aabb"`? `"leetcode"`?

**Bài 2 (tính tay):** với `"abcab"`, viết bảng trace của Cách 4 (`j`, `c`, `cnt[c]`, `ans`).

**Bài 3 (điền TODO):** hoàn thiện `prefix_answers(s)` trả về list `ans_k` = đáp án cho tiền tố `s[:k]`, `k = 1..n`, trong một lượt.

**Bài 4 (tự viết lại):** viết `total_length(s)` = **tổng độ dài** của mọi xâu con bắt đầu và kết thúc cùng ký tự, trong `O(n)`. (Biến thể tự đặt.)

**Bài 5 (mini challenge):** cho `s` và `queries[i] = [l, r]`; trả về đáp án của bài gốc cho `s[l..r]` với mỗi truy vấn. Yêu cầu `O(26)` mỗi truy vấn sau tiền xử lý `O(26 n)`. (Biến thể tự đặt — đối chiếu lại đề gốc nếu dùng làm đề.)

In [30]:
def prefix_answers(s):
    # TODO: một lượt, bộ đếm chạy, append ans sau mỗi ký tự
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) Đếm tần suất. (2) `banana`: a=3, n=2, b=1. (3) `6 + 3 + 1 = 10`; `aabb`: `3 + 3 = 6`; `leetcode`: e=3 ⇒ `6`, còn 5 chữ đơn ⇒ `11`.

**Bài 2:** (1) Phần cộng ở mỗi bước là `cnt[c]` sau khi tăng. (2) Dãy phần cộng: `1, 1, 1, 2, 2`. (3) Tổng `7`.

**Bài 3:** (1) Như Cách 4. (2) `res.append(ans)` trong vòng lặp. (3) Kiểm tra bằng `count_formula(s[:k])`.

**Bài 4:** (1) Xâu con `(i, j)` có độ dài `j - i + 1`. (2) Tại đầu phải `j`, tổng độ dài các xâu kết thúc ở `j` = `Σ_{i ∈ P, i ≤ j} (j - i + 1) = cnt·(j + 1) - sumpos`, với `cnt`, `sumpos` = số lượng và tổng vị trí của chữ `s[j]` tới `j` (tính cả `j`). (3) Giữ hai mảng 26 phần tử.

**Bài 5:** (1) `pre[k][c]` = số lần chữ `c` trong `s[0..k-1]`. (2) `f_c = pre[r+1][c] - pre[l][c]`. (3) Đáp án `Σ f_c(f_c+1)/2`. Bộ nhớ `26(n+1)`; có thể lưu 26 list prefix riêng.

## 19. Đáp án tham khảo

In [31]:
# Bài 1
assert [Solution().numberOfSubstrings(x) for x in ("banana", "aabb", "leetcode")] == [10, 6, 11]
assert [count_brute_substrings(x) for x in ("banana", "aabb", "leetcode")] == [10, 6, 11]

# Bài 2
assert trace_running("abcab") == 7

# Bài 3
def prefix_answers_ans(s):
    cnt, ans, res = [0] * 26, 0, []
    for c in s:
        k = ord(c) - 97; cnt[k] += 1; ans += cnt[k]; res.append(ans)
    return res
assert prefix_answers_ans("abacad") == [count_formula("abacad"[:k]) for k in range(1, 7)]

# Bài 4
def total_length_ans(s):
    cnt, sp, total = [0] * 26, [0] * 26, 0
    for j, c in enumerate(s):
        k = ord(c) - 97; cnt[k] += 1; sp[k] += j
        total += cnt[k] * (j + 1) - sp[k]
    return total

def total_length_brute(s):
    return sum(j - i + 1 for i in range(len(s)) for j in range(i, len(s)) if s[i] == s[j])

rng = random.Random(41)
for _ in range(500):
    s = "".join(rng.choice("abc") for _ in range(rng.randint(1, 25)))
    assert total_length_ans(s) == total_length_brute(s)
print("Bài 4: tổng độ dài 'abcba' =", total_length_ans("abcba"), "(1*5 + 3 + 5 = 13)")
print("Bài 1-4: OK")

s = 'abcab'
 j | c | cnt[c] | +  | ans | xâu con kết thúc tại j
 0 | a |      1 |  1 |   1 | ['a']
 1 | b |      1 |  1 |   2 | ['b']
 2 | c |      1 |  1 |   3 | ['c']
 3 | a |      2 |  2 |   5 | ['abca', 'a']
 4 | b |      2 |  2 |   7 | ['bcab', 'b']

Bài 4: tổng độ dài 'abcba' = 13 (1*5 + 3 + 5 = 13)
Bài 1-4: OK


In [32]:
# Bài 5 — truy vấn đoạn với prefix 26 chữ cái
def range_queries(s, queries):
    n = len(s)
    pre = [[0] * (n + 1) for _ in range(26)]
    for c in range(26):
        row = pre[c]; ch = chr(97 + c)
        for i, x in enumerate(s):
            row[i + 1] = row[i] + (x == ch)
    out = []
    for l, r in queries:
        tot = 0
        for c in range(26):
            f = pre[c][r + 1] - pre[c][l]
            tot += f * (f + 1) // 2
        out.append(tot)
    return out

rng = random.Random(55)
for _ in range(200):
    s = "".join(rng.choice("abcd") for _ in range(rng.randint(1, 30)))
    qs = []
    for _ in range(8):
        l = rng.randrange(len(s)); qs.append([l, rng.randrange(l, len(s))])
    assert range_queries(s, qs) == [count_pairs_n2(s[l:r + 1]) for l, r in qs]
print("Bài 5 ví dụ:", range_queries("abacad", [[0, 5], [0, 2], [1, 3], [3, 3]]), "(kỳ vọng [9, 4, 3, 1])")

Bài 5 ví dụ: [9, 4, 3, 1] (kỳ vọng [9, 4, 3, 1])


**Nhận xét về Bài 5:** tiền xử lý `26 × (n + 1)` số nguyên. Với `n = 10⁵` đó là `2.6·10⁶` phần tử list — chấp nhận được. Nếu số truy vấn rất lớn và chỉ cần đáp án, có thể lưu thêm "đáp án tiền tố" nhưng **không** trừ trực tiếp được: `ans(s[0..r]) - ans(s[0..l-1])` **sai**, vì các cặp `(i, j)` với `i < l ≤ j` bị tính vào `ans(s[0..r])` mà không thuộc đoạn. Kiểm chứng:

In [33]:
def wrong_prefix_subtract(s, l, r):
    pa = prefix_answers_ans(s)
    return pa[r] - (pa[l - 1] if l > 0 else 0)          # SAI: còn các cặp vắt qua ranh giới l

s_ = "abacad"
for l, r in [(1, 5), (2, 4), (0, 5)]:
    print(f"[{l},{r}]: trừ tiền tố = {wrong_prefix_subtract(s_, l, r)} | đúng = {range_queries(s_, [[l, r]])[0]}")

[1,5]: trừ tiền tố = 8 | đúng = 6
[2,4]: trừ tiền tố = 6 | đúng = 4
[0,5]: trừ tiền tố = 9 | đúng = 9


Hàm `f(f+1)/2` **không tuyến tính** theo tần suất, nên phải lấy tần suất đoạn trước rồi mới áp công thức — đó là lý do cần prefix **theo từng chữ** chứ không phải prefix của đáp án.

## 20. Template nhớ nhanh + Checklist

```python
# Đếm cặp (i <= j) có cùng giá trị: "tăng rồi cộng"
cnt = defaultdict(int); ans = 0
for x in seq:
    cnt[x] += 1
    ans += cnt[x]
# Cặp i < j: "cộng rồi tăng"  (ans += cnt[x]; cnt[x] += 1)
# Công thức đóng: sum(f * (f + 1) // 2 for f in Counter(seq).values())
```

**Checklist trước khi submit:**
- [ ] Tính cả xâu con độ dài 1 (`f(f+1)/2`, không phải `f(f-1)/2`).
- [ ] Thứ tự tăng/cộng đúng với `i ≤ j`.
- [ ] Đếm theo **vị trí**, không phải xâu phân biệt.
- [ ] Không tạo xâu con trong vòng lặp (`O(n³)`).
- [ ] Ở ngôn ngữ khác: kết quả tới `≈ 5·10⁹` ⇒ dùng `long`.